In [12]:
import json
import os
from typing import Literal, TypedDict
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field
from IPython.display import Image, display
load_dotenv()

True

In [6]:
Intent = Literal[
    "billing",
    "technical_issue",
    "account_access",
    "shipping",
    "general",
]

Priority = Literal["low", "medium", "high", "urgent"]


class Classification(BaseModel):
    intent: Intent = Field(description="Main customer intent")
    priority: Priority = Field(description="Support priority level")
    summary: str = Field(description="Short summary of the customer issue")
    reasoning: str = Field(description="Brief reason for the classification")


class SupportState(TypedDict, total=False):
    query: str
    top_k: int
    classification: Classification
    retrieved_passages: list[dict]
    draft_response: str
    final_response: str
    trace: list[dict]

In [7]:
KB_DOCUMENTS = [
    Document(
        page_content=(
            "Duplicate billing: If a customer sees two charges for the same invoice, "
            "verify the transaction IDs and billing dates. If confirmed, issue a refund "
            "within 5 to 7 business days and notify the customer by email."
        ),
        metadata={
            "article_id": "KB-BILL-001",
            "title": "Duplicate Charges and Refunds",
            "intent": "billing",
        },
    ),
    Document(
        page_content=(
            "Failed payment: Ask the customer to confirm card expiry, billing address, "
            "available balance, and whether their bank blocked the transaction. Customers "
            "can update payment details from Settings > Billing > Payment Methods."
        ),
        metadata={
            "article_id": "KB-BILL-002",
            "title": "Failed Payment Troubleshooting",
            "intent": "billing",
        },
    ),
    Document(
        page_content=(
            "Subscription cancellation: Customers can cancel from Settings > Billing > Plan. "
            "Access remains active until the end of the current billing period."
        ),
        metadata={
            "article_id": "KB-BILL-003",
            "title": "Cancel a Subscription",
            "intent": "billing",
        },
    ),
    Document(
        page_content=(
            "Login issues: If the customer cannot sign in, ask them to reset their password. "
            "If they do not receive the reset email, check spam folders and confirm the email "
            "address on the account."
        ),
        metadata={
            "article_id": "KB-ACC-001",
            "title": "Unable to Sign In",
            "intent": "account_access",
        },
    ),
    Document(
        page_content=(
            "Account lockout: Accounts are temporarily locked after repeated failed login "
            "attempts. The customer should wait 30 minutes or use the password reset flow. "
            "Escalate if the customer suspects unauthorized access."
        ),
        metadata={
            "article_id": "KB-ACC-002",
            "title": "Account Locked After Failed Attempts",
            "intent": "account_access",
        },
    ),
    Document(
        page_content=(
            "Password reset: Customers can reset their password using the Forgot Password link "
            "on the login page. The reset link expires after 20 minutes."
        ),
        metadata={
            "article_id": "KB-ACC-003",
            "title": "Password Reset Process",
            "intent": "account_access",
        },
    ),
    Document(
        page_content=(
            "Mobile app crash: Ask the customer for device type, operating system, app version, "
            "time of crash, and screenshots if available. Recommend updating the app and "
            "clearing cache before escalating to engineering."
        ),
        metadata={
            "article_id": "KB-TECH-001",
            "title": "Mobile App Crash Troubleshooting",
            "intent": "technical_issue",
        },
    ),
    Document(
        page_content=(
            "API errors: For 500-level errors, collect request ID, endpoint, timestamp, "
            "payload sample, and account ID. Check status page incidents before escalating."
        ),
        metadata={
            "article_id": "KB-TECH-002",
            "title": "API Error Investigation",
            "intent": "technical_issue",
        },
    ),
    Document(
        page_content=(
            "Slow dashboard loading: Ask the customer to confirm browser version, network "
            "conditions, affected dashboard name, and approximate load time. Recommend clearing "
            "browser cache and trying an incognito window."
        ),
        metadata={
            "article_id": "KB-TECH-003",
            "title": "Dashboard Performance Troubleshooting",
            "intent": "technical_issue",
        },
    ),
    Document(
        page_content=(
            "Delayed shipment: Check the carrier tracking number and estimated delivery date. "
            "If tracking has not updated for more than 72 hours, open a carrier investigation."
        ),
        metadata={
            "article_id": "KB-SHIP-001",
            "title": "Delayed Shipment Process",
            "intent": "shipping",
        },
    ),
    Document(
        page_content=(
            "Lost shipment: If the carrier marks a package as delivered but the customer cannot "
            "find it, ask them to check nearby areas, household members, and building reception. "
            "If still missing after 24 hours, start a lost package claim."
        ),
        metadata={
            "article_id": "KB-SHIP-002",
            "title": "Lost Shipment Investigation",
            "intent": "shipping",
        },
    ),
]

In [8]:
SAMPLE_QUERIES = [
    "I was charged twice this month for the same subscription. Can you fix this?",
    "I cannot log in even after trying my password several times. Is my account blocked?",
    "Your mobile app keeps crashing whenever I open the reports page.",
    "My order tracking has not updated in four days. Where is my package?",
    "The API is returning 500 errors for our checkout requests. This is affecting customers.",
]

In [9]:
def add_trace(state: SupportState, step: str, data: dict) -> list[dict]:
    return state.get("trace", []) + [{"step": step, "data": data}]


def build_vector_store() -> InMemoryVectorStore:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(KB_DOCUMENTS)

    return vector_store

In [17]:
def build_graph(llm: ChatAnthropic, vector_store: InMemoryVectorStore):
    classifier_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "You are a customer support classifier. "
                    "Classify the customer message into one intent and one priority. "
                    "Intent must be one of: billing, technical_issue, account_access, "
                    "shipping, general. "
                    "Priority must be one of: low, medium, high, urgent. "
                    "Use urgent only for business-critical outages, security risks, "
                    "or many customers being impacted."
                ),
            ),
            ("human", "Customer message: {query}"),
        ]
    )

    composer_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "You are a professional customer support agent. "
                    "Use the provided KB passages to draft a helpful response. "
                    "Be concise, empathetic, and specific. "
                    "Do not invent policies that are not in the KB."
                ),
            ),
            (
                "human",
                (
                    "Customer query:\n{query}\n\n"
                    "Classification:\n{classification}\n\n"
                    "Relevant KB passages:\n{kb_context}\n\n"
                    "Draft the customer response."
                ),
            ),
        ]
    )

    finalizer_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                (
                    "Polish this support draft into a final customer-facing response. "
                    "Remove internal labels, KB article IDs, priority labels, and reasoning. "
                    "Keep the response professional, concise, and actionable."
                ),
            ),
            ("human", "{draft_response}"),
        ]
    )

    def classifier_agent(state: SupportState) -> SupportState:
        structured_llm = llm.with_structured_output(Classification)

        classification = structured_llm.invoke(
            classifier_prompt.format_messages(query=state["query"])
        )

        return {
            **state,
            "classification": classification,
            "trace": add_trace(
                state,
                "classifier_agent",
                classification.model_dump(),
            ),
        }

    def retrieval_agent(state: SupportState) -> SupportState:
        classification = state["classification"]
        top_k = state.get("top_k", 3)

        search_query = (
            f"Intent: {classification.intent}. "
            f"Priority: {classification.priority}. "
            f"Issue summary: {classification.summary}. "
            f"Original customer message: {state['query']}"
        )

        docs = vector_store.similarity_search(search_query, k=top_k)

        passages = [
            {
                "article_id": doc.metadata["article_id"],
                "title": doc.metadata["title"],
                "intent": doc.metadata["intent"],
                "passage": doc.page_content,
            }
            for doc in docs
        ]

        return {
            **state,
            "retrieved_passages": passages,
            "trace": add_trace(
                state,
                "retrieval_agent",
                {
                    "search_query": search_query,
                    "top_k": top_k,
                    "passages": passages,
                },
            ),
        }

    def composer_agent(state: SupportState) -> SupportState:
        classification = state["classification"].model_dump()

        kb_context = "\n\n".join(
            [
                f"Title: {item['title']}\nPassage: {item['passage']}"
                for item in state["retrieved_passages"]
            ]
        )

        draft_response = llm.invoke(
            composer_prompt.format_messages(
                query=state["query"],
                classification=json.dumps(classification, indent=2),
                kb_context=kb_context,
            )
        ).content

        final_response = llm.invoke(
            finalizer_prompt.format_messages(draft_response=draft_response)
        ).content

        return {
            **state,
            "draft_response": draft_response,
            "final_response": final_response,
            "trace": add_trace(
                state,
                "composer_agent",
                {
                    "draft_response": draft_response,
                    "final_response": final_response,
                },
            ),
        }

    graph = StateGraph(SupportState)

    graph.add_node("classifier_agent", classifier_agent)
    graph.add_node("retrieval_agent", retrieval_agent)
    graph.add_node("composer_agent", composer_agent)

    graph.set_entry_point("classifier_agent")
    graph.add_edge("classifier_agent", "retrieval_agent")
    graph.add_edge("retrieval_agent", "composer_agent")
    graph.add_edge("composer_agent", END)
    
    return graph.compile()


In [18]:
llm = ChatAnthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL"),
    base_url=os.environ.get("ENDPOINT"),
    temperature=0,
    max_tokens=256
)

vector_store = build_vector_store()
support_triage_graph = build_graph(llm, vector_store)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7067.36it/s]


In [20]:
results = []

for index, query in enumerate(SAMPLE_QUERIES, start=1):
    result = support_triage_graph.invoke(
        {
            "query": query,
            "top_k": 3,
            "trace": [],
        }
    )

    results.append(
        {
            "sample_number": index,
            "customer_query": result["query"],
            "classification_result": result["classification"].model_dump(),
            "retrieved_kb_passages": result["retrieved_passages"],
            "draft_response_generation": result["draft_response"],
            "final_customer_facing_response": result["final_response"],
            "agent_execution_trace": result["trace"],
        }
    )

In [22]:
for item in results:
    print("=" * 100)
    print(f"SAMPLE QUERY {item['sample_number']}")
    print("=" * 100)

    print("\nCUSTOMER QUERY")
    print(item["customer_query"])

    print("\nCLASSIFICATION RESULT")
    print(json.dumps(item["classification_result"], indent=2))

    print("\nRETRIEVED KB PASSAGES")
    print(json.dumps(item["retrieved_kb_passages"], indent=2))

    print("\nDRAFT RESPONSE GENERATION")
    print(item["draft_response_generation"])

    print("\nFINAL CUSTOMER-FACING RESPONSE")
    print(item["final_customer_facing_response"])

    print("\nAGENT EXECUTION TRACE")
    print(json.dumps(item["agent_execution_trace"], indent=2))

    print("\n")

SAMPLE QUERY 1

CUSTOMER QUERY
I was charged twice this month for the same subscription. Can you fix this?

CLASSIFICATION RESULT
{
  "intent": "billing",
  "priority": "high",
  "summary": "Customer was charged twice for the same subscription in the current month and requests a refund/correction",
  "reasoning": "This is a billing issue involving duplicate charges, which directly impacts the customer's finances. While not an outage or security breach affecting many customers, duplicate charges warrant high priority as they require immediate investigation and resolution to prevent customer dissatisfaction and potential chargeback disputes."
}

RETRIEVED KB PASSAGES
[
  {
    "article_id": "KB-BILL-001",
    "title": "Duplicate Charges and Refunds",
    "intent": "billing",
    "passage": "Duplicate billing: If a customer sees two charges for the same invoice, verify the transaction IDs and billing dates. If confirmed, issue a refund within 5 to 7 business days and notify the customer b